# Streaming from Kafka with LakeLogic

A real Kafka broker (Redpanda, in Docker), real events, and the cases that decide whether a stream can be trusted:

| Case | What must hold |
|---|---|
| Drain the topic | good rows land in bronze; bad rows land in quarantine **with the reason** |
| Run again later | only the **new** events are read |
| Crash mid-run, restart | no gap and no duplicate |
| Replay the whole topic | the `merge` target still has one row per ride |
| A message that is not JSON | it is quarantined; the stream does **not** stop |
| The same ride updated twice in one batch | the **last** version wins; the earlier one is counted as `superseded` |

**Start the broker** (once):

```
docker run -d --name lakelogic-kafka-dev -p 19092:19092 docker.redpanda.com/redpandadata/redpanda:v24.2.7 \
  redpanda start --mode dev-container --smp 1 \
  --kafka-addr internal://0.0.0.0:9092,external://0.0.0.0:19092 \
  --advertise-kafka-addr internal://localhost:9092,external://localhost:19092
```

Needs `pip install "lakelogic[kafka]" deltalake`. Set `ENGINE` below to `polars`, `duckdb` or `spark`.

In [1]:
import os
import sys
from pathlib import Path

LAKELOGIC_SRC = None  # a local lakelogic checkout, if you are not using the released package
if LAKELOGIC_SRC:
    sys.path.insert(0, LAKELOGIC_SRC)
os.environ.setdefault("ENGINE", "polars")  # polars | duckdb | spark
os.environ.setdefault("KAFKA_BROKERS", "localhost:19092")

import polars as pl
from loguru import logger
from kafka.admin import KafkaAdminClient

logger.remove()  # quiet; add a sink to watch the runs
pl.Config.set_tbl_rows(12)
pl.Config.set_fmt_str_lengths(90)
pl.Config.set_tbl_width_chars(200)

import kafka_scenarios as K  # the contract, the event generator and the checks live here

print("broker:", KafkaAdminClient(bootstrap_servers=K.BROKERS).list_topics()[:3], "... reachable")
print("engine:", K.ENGINE)

broker: ['smoke', 'rides_polars_drain_5c6610', 'rides_polars_resume_65205a'] ... reachable
engine: polars


## Run it from the contract file

Everything the run needs is in [`contracts/rides_kafka.yaml`](contracts/rides_kafka.yaml): the broker, the topic,
where the offsets are kept, the schema, the rule, and where good and bad rows go. No Python wiring:
`DataProcessor(contract=...).run_source()` (or the pipeline runner) drains the topic, writes every micro-batch,
and commits the offsets after each write.

In [2]:
print(Path("contracts/rides_kafka.yaml").read_text())

# Bronze rides from a Kafka topic. Everything the run needs is here: the broker, the topic,
# where the offsets are kept, the schema, the rules, and where good and bad rows go.
version: 1.0.0
dataset: rides
info: {title: bronze_rides_kafka}
primary_key: [ride_id]
source:
  type: stream
  options:
    kind: kafka
    brokers: env:KAFKA_BROKERS              # e.g. localhost:19092 (Redpanda in Docker)
    topic: rides
    starting_offsets: earliest
    checkpoint: data/checkpoints/rides_kafka.sqlite   # Polars/DuckDB offsets; Spark keeps its own in rides_kafka_spark/
    batch_size: 100
    trigger: available_now                  # drain to the current end, then stop; `continuous` keeps running
    # A secured cluster (SASL): the password is always an environment variable, never a literal.
    # security_protocol: SASL_SSL
    # sasl_mechanism: PLAIN
    # sasl_username: my-user
    # sasl_password: env:KAFKA_PASSWORD
model:
  fields:
    - {name: ride_id, type: long, required: true}
    

In [3]:
import shutil
from lakelogic import DataProcessor

for d in ("data/bronze", "data/quarantine", "data/checkpoints"):
    shutil.rmtree(d, ignore_errors=True)
K.reset_topic("rides")
K.produce("rides", [K.ride(i) for i in range(100)] + [K.ride(900 + i, fare=-1.0) for i in range(5)] + [b"not json"])


def run_contract(path="contracts/rides_kafka.yaml"):
    s = DataProcessor(engine=K.ENGINE, contract=path).run_source().stream_summary
    print(
        f"read {s['source_count']} events: {s['good_count']} good, {s['bad_count']} quarantined, "
        f"{s['superseded_count']} replaced by a later event for the same ride"
    )


run_contract()  # 100 good; 5 negative fares + 1 non-JSON quarantined
K.produce("rides", [K.ride(i) for i in range(200, 210)])
run_contract()  # only the 10 new events
run_contract()  # nothing new

read 106 events: 100 good, 6 quarantined, 0 replaced by a later event for the same ride


read 10 events: 10 good, 0 quarantined, 0 replaced by a later event for the same ride
read 0 events: 0 good, 0 quarantined, 0 replaced by a later event for the same ride


### The same contract on Azure Event Hubs

[`contracts/rides_eventhubs.yaml`](contracts/rides_eventhubs.yaml) changes only the source: `kind: eventhubs` and
`connection_string: env:EVENTHUBS_CONNECTION_STRING`. The brokers and the SASL login come from that connection
string, which stays in the environment. Needs an event hub named `rides`. Skipped when the variable is not set.

An event hub keeps events for its retention period and cannot be emptied, so a fresh checkpoint also reads
events sent by earlier runs. The `merge` still leaves one row per ride: a ride sent twice is counted as replaced.

In [4]:
print(Path("contracts/rides_eventhubs.yaml").read_text())

# Bronze rides from an Azure Event Hub. Same contract as rides_kafka.yaml; only the source differs:
# the namespace connection string (an environment variable) gives the brokers and the SASL login.
version: 1.0.0
dataset: rides
info: {title: bronze_rides_eventhubs}
primary_key: [ride_id]
source:
  type: stream
  options:
    kind: eventhubs                         # Azure Event Hubs, through its Kafka endpoint (port 9093)
    connection_string: env:EVENTHUBS_CONNECTION_STRING   # namespace connection string; the brokers come from it
    topic: rides                            # the event hub name
    starting_offsets: earliest
    checkpoint: data/checkpoints/rides_eventhubs.sqlite
    batch_size: 100
    trigger: available_now
model:
  fields:
    - {name: ride_id, type: long, required: true}
    - {name: city, type: string, required: true}
    - {name: fare, type: double, required: true}
    - {name: status, type: string}
quality:
  row_rules:
    - {name: fare_not_negative, sql: "fa

In [5]:
cs = os.environ.get("EVENTHUBS_CONNECTION_STRING")
if cs:
    for d in ("data/bronze", "data/quarantine", "data/checkpoints"):
        shutil.rmtree(d, ignore_errors=True)
    eh = {
        "security_protocol": "SASL_SSL",
        "sasl_mechanism": "PLAIN",
        "sasl_plain_username": "$ConnectionString",
        "sasl_plain_password": cs,
    }
    host = cs.split("sb://")[1].split("/")[0] + ":9093"
    K.produce("rides", [K.ride(i) for i in range(50)] + [K.ride(900, fare=-1.0)], auth=eh, brokers=host)
    run_contract("contracts/rides_eventhubs.yaml")
    run_contract("contracts/rides_eventhubs.yaml")  # nothing new
    import deltalake

    print("rows in bronze:", deltalake.DeltaTable("data/bronze").to_pyarrow_table().num_rows)
else:
    print("EVENTHUBS_CONNECTION_STRING not set — skipped")

read 156 events: 100 good, 5 quarantined, 51 replaced by a later event for the same ride


read 0 events: 0 good, 0 quarantined, 0 replaced by a later event for the same ride
rows in bronze: 50


## Under the hood: the hard cases, step by step

The sections below drive the same machinery through the Python API (`StreamSink` + `KafkaOffsetSource`),
each on its own topic, to show the cases that decide whether a stream can be trusted.

## 1. Drain the topic: good rows to bronze, bad rows to quarantine

500 good rides, 20 with a negative fare, 10 with no `ride_id`, sent to a topic with 3 partitions.
`available_now` reads everything that is there, then stops.

In [6]:
r = K.Run("walkthrough")
r.send(
    [K.ride(i) for i in range(500)]
    + [K.ride(1000 + i, fare=-5.0) for i in range(20)]
    + [{"city": "Rome", "fare": 9.0} for _ in range(10)]
)

s = r.sink().run("available_now")
print(f"read {s.source_count} events in {s.batches} batches: {s.good_count} good, {s.bad_count} quarantined")
print("checkpoint (next offset per partition):", s.cursor)
display(r.table().sort("ride_id").head(5))

read 530 events in 6 batches: 500 good, 30 quarantined
checkpoint (next offset per partition): {'rides_polars_walkthrough_56cb92:0': 147, 'rides_polars_walkthrough_56cb92:1': 177, 'rides_polars_walkthrough_56cb92:2': 206}


ride_id,city,fare,status
i64,str,f64,str
0,"""London""",5.0,"""completed"""
1,"""Paris""",6.0,"""completed"""
2,"""Lagos""",7.0,"""completed"""
3,"""London""",8.0,"""completed"""
4,"""Paris""",9.0,"""completed"""


In [7]:
q = r.quarantine()
print(q.height, "rows in quarantine. Why:")
display(q.select("ride_id", "city", "fare", "_lakelogic_errors").head(6))

30 rows in quarantine. Why:


ride_id,city,fare,_lakelogic_errors
i64,str,f64,list[str]
1000,"""Paris""",-5.0,"[""Rule failed: fare_not_negative (fare >= 0)""]"
1004,"""Lagos""",-5.0,"[""Rule failed: fare_not_negative (fare >= 0)""]"
1007,"""Lagos""",-5.0,"[""Rule failed: fare_not_negative (fare >= 0)""]"
1008,"""London""",-5.0,"[""Rule failed: fare_not_negative (fare >= 0)""]"
1009,"""Paris""",-5.0,"[""Rule failed: fare_not_negative (fare >= 0)""]"
1012,"""Paris""",-5.0,"[""Rule failed: fare_not_negative (fare >= 0)""]"


## 2. Run again later: only the new events are read

The checkpoint stores the next offset of every partition, committed **after** each write.

In [8]:
r.send([K.ride(i) for i in range(2000, 2060)])
s2 = r.sink().run("available_now")
print("second run read", s2.source_count, "events (the 60 new ones)")
s3 = r.sink().run("available_now")
print("third run read", s3.source_count, "events (nothing new)")
print("rows in bronze:", r.table().height)

second run read 60 events (the 60 new ones)
third run read 0 events (nothing new)
rows in bronze: 560


## 3. A message that is not JSON

One bad message used to stop the stream for good: it failed to decode, the run died before its checkpoint,
and every restart hit the same message again.
Now it is quarantined with the partition, the offset and its first bytes, and the stream carries on.

In [9]:
r.send([b"{not json", b"\xff\xfe"], raw=True)
r.send([K.ride(i) for i in range(3000, 3010)])
s4 = r.sink().run("available_now")
print(f"read {s4.source_count}: {s4.good_count} good, {s4.bad_count} quarantined")
q = r.quarantine()
display(
    q.filter(pl.col("ride_id").is_null() & pl.col("city").is_null()).select(
        pl.col("_lakelogic_errors").list.first().alias("reason")
    )
)

read 12: 10 good, 2 quarantined


reason
str
"""Message is not valid JSON (JSONDecodeError) (partition 0, offset 168): '{not json'"""
"""Message is not valid JSON (JSONDecodeError) (partition 0, offset 169): '��'"""


## 4. The same ride updated twice in one batch

Ride 3000 completes, then is refunded, and both events arrive in the same micro-batch.
The last event wins; the earlier one is counted as `superseded`, so the numbers still add up:
`read = good + quarantined + superseded`.

In [10]:
r.send([K.ride(3000, status="refunded", fare=0.0)])  # 3000 was written above as completed
r.send([K.ride(4000), K.ride(4000, status="refunded", fare=0.0)])  # both in one batch
s5 = r.sink().run("available_now")
print(f"read {s5.source_count}: good {s5.good_count}, superseded {s5.superseded_count}")
display(r.table().filter(pl.col("ride_id").is_in([3000, 4000])))

read 3: good 2, superseded 1


ride_id,city,fare,status
i64,str,f64,str
3000,"""London""",0.0,"""refunded"""
4000,"""Paris""",5.0,"""completed"""


## 5. Replay everything: the merge keeps one row per ride

A new checkpoint key reads the topic from the start again.
Every event is processed a second time and the table does not grow.

In [11]:
before = r.table().height
replay = r.sink(key="replay_everything").run("available_now")
after = r.table()
print(f"replayed {replay.source_count} events; bronze rows before {before}, after {after.height}")
print("one row per ride:", after["ride_id"].n_unique() == after.height)

replayed 605 events; bronze rows before 571, after 571
one row per ride: True


## Scenario suite

Every case above and the crash cases, each on its own topic, checked automatically.
Run it with `ENGINE=duckdb` or `ENGINE=spark` to see the same answers on the other engines.

In [12]:
K.RESULTS.clear()
for name, fn, expect in K.ALL:
    K.scenario(name, fn, **expect)
display(K.summary().select("status", "scenario", "result", "why"))

[PASS] drain: 500 good, 20 negative fares, 10 missing ride_id     consumed=530, good=500, quarantined=30, rows_in_bronze=500, rows_in_quarantine=30


[PASS] resume: a later run reads only the new events              first_run=200, second_run=60, third_run=0, rows_in_bronze=260


[PASS] crash after one batch, then resume                         resumed_and_read=200, rows_in_bronze=300, no_duplicates=True


[PASS] crash between the write and the checkpoint commit          reread_after_crash=200, rows_in_bronze=300, no_duplicates=True


[PASS] replay the whole topic: merge keeps one row per ride       replayed=150, rows_in_bronze=150, no_duplicates=True


[PASS] two messages are not JSON                                  consumed=102, good=100, quarantined=2, rows_in_bronze=100, rows_in_quarantine=2, reason_says_not_json=True


[PASS] wrong type in a field; a new field appears                 good=45, quarantined=5, rows_in_bronze=45


[PASS] the same ride updated later in the same batch              consumed=40, superseded=10, rows_in_bronze=30, refunded=10, no_duplicates=True

8 of 8 streaming scenarios behaved as expected (polars).


status,scenario,result,why
str,str,str,str
"""PASS""","""drain: 500 good, 20 negative fares, 10 missing ride_id""","""consumed=530, good=500, quarantined=30, rows_in_bronze=500, rows_in_quarantine=30""",""""""
"""PASS""","""resume: a later run reads only the new events""","""first_run=200, second_run=60, third_run=0, rows_in_bronze=260""",""""""
"""PASS""","""crash after one batch, then resume""","""resumed_and_read=200, rows_in_bronze=300, no_duplicates=True""",""""""
"""PASS""","""crash between the write and the checkpoint commit""","""reread_after_crash=200, rows_in_bronze=300, no_duplicates=True""",""""""
"""PASS""","""replay the whole topic: merge keeps one row per ride""","""replayed=150, rows_in_bronze=150, no_duplicates=True""",""""""
"""PASS""","""two messages are not JSON""","""consumed=102, good=100, quarantined=2, rows_in_bronze=100, rows_in_quarantine=2, reason_sa…",""""""
"""PASS""","""wrong type in a field; a new field appears""","""good=45, quarantined=5, rows_in_bronze=45""",""""""
"""PASS""","""the same ride updated later in the same batch""","""consumed=40, superseded=10, rows_in_bronze=30, refunded=10, no_duplicates=True""",""""""
